# Mibian Black–Scholes IV and Greeks (NIFTY example)

This notebook demonstrates the use of the `mibian` library to:
- Install the `mibian` package from PyPI.
- Solve for implied volatility (IV) from given market option prices.
- Compute option Greeks (Delta, Gamma, Theta, Vega) at the solved IV.
- Use calendar days and percentage inputs, consistent with Mibian's requirements.

In [ ]:
# 2) Setup: Install mibian quietly
import sys
!{sys.executable} -m pip install mibian --quiet

In [30]:
from datetime import datetime, timezone
from typing import Optional
import mibian

def days_to_expiry(expiry_date_str: str, today_str: Optional[str] = None) -> int:
    """
    Calculates the number of calendar days from today to an expiry date.
    Mibian expects the time to expiry in calendar days.
    """
    expiry_date = datetime.strptime(expiry_date_str, '%Y-%m-%d').date()
    if today_str:
        today = datetime.strptime(today_str, '%Y-%m-%d').date()
    else:
        today = datetime.now(timezone.utc).date()
    return max(0, (expiry_date - today).days)

def format_float(value: float) -> str:
    """Formats a float to 4 decimal places."""
    return f"{value:.4f}"

def format_pct(value: float) -> str:
    """Formats a float as a percentage with 2 decimal places."""
    return f"{value:.2f}%"# 3) Imports and helpers


In [34]:
# we first we calculate synthetic futures here.

# 4) Inputs
synth_future = 24702.85 # here we add synthetic future price
call_strike = 24650
put_strike = 24650
call_ltp = 113.25
put_ltp = 65.1
risk_free_rate = 0.0  # Using syntetic futures as underlying (Black-76 style)
expiry_days = 3     # Demo: days_to_expiry('2020-03-30', '2020-02-28')

print("--- Inputs ---")
print(f"synth_future: {synth_future}")
print(f"Call Strike:      {call_strike}")
print(f"Put Strike:       {put_strike}")
print(f"Call LTP:         {call_ltp}")
print(f"Put LTP:          {put_ltp}")
print(f"Risk-Free Rate:   {format_pct(risk_free_rate)}")
print(f"Days to Expiry:   {expiry_days}")

--- Inputs ---
synth_future: 24702.85
Call Strike:      24650
Put Strike:       24650
Call LTP:         113.25
Put LTP:          65.1
Risk-Free Rate:   0.00%
Days to Expiry:   3


In [35]:
# 5) Solve implied vols
call_bs = mibian.BS([synth_future, call_strike, risk_free_rate, expiry_days], callPrice=call_ltp)
call_iv = call_bs.impliedVolatility
print(f"Call IV (percent): {format_pct(call_iv)}")

put_bs = mibian.BS([synth_future, put_strike, risk_free_rate, expiry_days], putPrice=put_ltp)
put_iv = put_bs.impliedVolatility
print(f"Put IV (percent):  {format_pct(put_iv)}")

Call IV (percent): 9.43%
Put IV (percent):  9.98%


In [33]:
# 6) Compute Greeks at solved vols
call_greeks = mibian.BS([synth_future, call_strike, risk_free_rate, expiry_days], volatility=call_iv)
put_greeks = mibian.BS([synth_future, put_strike, risk_free_rate, expiry_days], volatility=put_iv)

print("--- Call Greeks (at solved IV) ---")
print(f"Model Call Price: {format_float(call_greeks.callPrice)}")
print(f"Call Delta:       {format_float(call_greeks.callDelta)}")
print(f"Call Theta:       {format_float(call_greeks.callTheta)} (per calendar day)")
print(f"Vega:             {format_float(call_greeks.vega)} (per 1% vol point)")
print(f"Gamma:            {format_float(call_greeks.gamma)}")

print("\n--- Put Greeks (at solved IV) ---")
print(f"Model Put Price:  {format_float(put_greeks.putPrice)}")
print(f"Put Delta:        {format_float(put_greeks.putDelta)}")
print(f"Put Theta:        {format_float(put_greeks.putTheta)} (per calendar day)")
print(f"Vega:             {format_float(put_greeks.vega)} (per 1% vol point)")
print(f"Gamma:            {format_float(put_greeks.gamma)}")

--- Call Greeks (at solved IV) ---
Model Call Price: 113.2523
Call Delta:       0.6005
Call Theta:       -20.4005 (per calendar day)
Vega:             7.0622 (per 1% vol point)
Gamma:            0.0018

--- Put Greeks (at solved IV) ---
Model Put Price:  65.1249
Put Delta:        -0.4047
Put Theta:        -21.6511 (per calendar day)
Vega:             7.0858 (per 1% vol point)
Gamma:            0.0017


In [22]:
# 7) Validation
tolerance = 0.10

call_price_repro = call_greeks.callPrice
call_error = abs(call_price_repro - call_ltp)
call_pass = call_error <= tolerance

put_price_repro = put_greeks.putPrice
put_error = abs(put_price_repro - put_ltp)
put_pass = put_error <= tolerance

print("--- Validation ---")
print(f"Call Re-priced: {format_float(call_price_repro)}, Market: {call_ltp}, Abs Error: {format_float(call_error)} -> {'PASS' if call_pass else 'FAIL'}")
print(f"Put Re-priced:  {format_float(put_price_repro)}, Market: {put_ltp}, Abs Error: {format_float(put_error)} -> {'PASS' if put_pass else 'FAIL'}")

print(f"\nValidation Summary: {'PASS' if call_pass and put_pass else 'FAIL'}")

--- Validation ---
Call Re-priced: 113.2468, Market: 113.25, Abs Error: 0.0032 -> PASS
Put Re-priced:  65.1231, Market: 65.1, Abs Error: 0.0231 -> PASS

Validation Summary: PASS


In [ ]:
# 8) Helper demo
expiry_str = '2020-03-30'
today_str = '2020-02-28'
days = days_to_expiry(expiry_str, today_str)
print(f"Days from {today_str} to {expiry_str}: {days} calendar days.")
print("Mibian uses calendar days for expiry, and its Theta is per calendar day.")

# 9) Notes and Caveats

- The `mibian` library expects inputs for risk-free rate and volatility as percentages (e.g., 5.0 for 5%).
- Using the futures price as the underlying with a zero risk-free rate is a common shortcut for pricing options on futures, corresponding to the Black-76 model.
- The implied volatilities for puts and calls often differ due to volatility smile/skew and other market factors not captured by the simple Black-Scholes model.
- This notebook is for educational and testing purposes only and does not constitute trading advice.